<a href="https://colab.research.google.com/github/mkeaudric/SDG-Visualizer/blob/main/Pake_model_bahasa_kecil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install -q sentence-transformers

In [23]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [24]:
model = SentenceTransformer(
    "intfloat/multilingual-e5-small"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Buat bobot masing-masing keyword SDG yang ada di elsevier:
$$W_i=\frac{\text{Ukuran Font}_i}{500}\times100\%$$


In [25]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("intfloat/multilingual-e5-small")

judul = "ANALISIS PENERAPAN COST LEADERSHIP STRATEGY DI CV. MEKAR MANDIRI BANDUNG"

# misal SDG1, dari rumus dapet:
sdg1 = {
    "Poverty": 100,
    "Income": 76,
    "Effect": 70.6,
    "Policy": 70.2,
    "Inequality": 69.8,
    "Public Policy": 61.8,
    "Human": 61.6,
    "Result": 60,
    "Impact": 59.8,
    "Analysis": 57.6,
    "Women": 55.6,
    "Economics": 55.2,
    "Child": 55.2,
    "Developing Countries": 32,
    "Development": 30.4,
    "Employment": 24.8
}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Kemiripan judul dan masing-masing keyword SDG:
$$
Sim(D,K_i) =
\frac{
\mathbf{v}_D \cdot \mathbf{v}_{K_i}
}{
\|\mathbf{v}_D\| \times \|\mathbf{v}_{K_i}\|
}
$$

In [26]:
# Judul jurnal jadi vektor
embedding_judul = model.encode(
    "passage: " + judul,
    normalize_embeddings=True
)

# Keyword (masing-masing) SDG jadi vektor
keywords = list(sdg1.keys())

embedding_keywords = model.encode(
    ["query: " + k for k in keywords],
    normalize_embeddings=True
)

# Hitung Cosine Similarity (ini keluarannya list btw)
similarities = cosine_similarity(
    [embedding_judul],
    embedding_keywords
)[0]

Kemiripan judul ke SDG:
$$
Score(D,SDG_j) =
\frac{
\displaystyle\sum_{i=1}^{n} W_i \times Sim(D,K_i)
}{
\displaystyle\sum_{i=1}^{n} W_i
}
$$

In [27]:
# Hitung weighted similarity
hasil = []

for keyword, similarity in zip(keywords, similarities):

    weight = sdg1[keyword] / 100
    score = weight * similarity

    hasil.append({
        "Keyword": keyword,
        "Bobot": weight,
        "Similarity": similarity,
        "Weighted Score": score
    })

df = pd.DataFrame(hasil)

final_score = (
    df["Weighted Score"].sum() /
    df["Bobot"].sum()
)

print(df.to_string(index=False))
print("\nFinal SDG 1 Score:", round(final_score, 4))

             Keyword  Bobot  Similarity  Weighted Score
             Poverty  1.000    0.813263        0.813263
              Income  0.760    0.830748        0.631368
              Effect  0.706    0.805635        0.568778
              Policy  0.702    0.818944        0.574898
          Inequality  0.698    0.816842        0.570156
       Public Policy  0.618    0.817306        0.505095
               Human  0.616    0.804823        0.495771
              Result  0.600    0.838027        0.502816
              Impact  0.598    0.792623        0.473988
            Analysis  0.576    0.851379        0.490394
               Women  0.556    0.788022        0.438140
           Economics  0.552    0.826457        0.456204
               Child  0.552    0.778688        0.429836
Developing Countries  0.320    0.798468        0.255510
         Development  0.304    0.805982        0.245019
          Employment  0.248    0.804633        0.199549

Final SDG 1 Score: 0.8134
